## Configurar ZAP
### Escaneo en busca de vulnerabilidades:
    - Comprobar comunicación con el frontent
    - Configura Firefox para usar el archivo PAC de ZAP.
    - Verifica que el proxy esté funcionando visitando un sitio web externo.
    - Escanear en frontend.
    - Visualizar reusltados

### Comprobar comunicación con frontend:

In [ ]:
docker exec -it security curl http://frontend:3001

### Configura Firefox para usar el archivo PAC de ZAP.
    - Hacer clic en el PAC file en el navegador localhost:80801
1. Abre Firefox.
2. Ve a Configuración (escribe about:preferences en la barra de direcciones).
3. Desplázate hacia abajo hasta la sección Configuración de red y haz clic en Configuración....
4. Selecciona Configuración automática de proxy.
5. En el campo URL del archivo de configuración automática de proxy (PAC), ingresa la URL del archivo PAC que ZAP te proporcionó:

In [ ]:
http://localhost:8081/OTHER/network/other/proxy.pac/?apinonce=2fba294e6c4744f1

### Reiniciar ZAP dentro del contenedor

### Verificar que el proxy esté funcionando
Para asegurarte de que Firefox esté enviando el tráfico a través de ZAP:

Abre ZAP en modo GUI.

Ve a la pestaña Sites.

Abre una nueva pestaña en Firefox y visita cualquier sitio web (por ejemplo, http://example.com).

Verifica en ZAP (pestaña Sites) si el tráfico de http://example.com aparece en el árbol de URLs.:

- Si aparece, significa que el proxy está funcionando correctamente.

- Si no aparece, revisa la configuración del proxy en Firefox.

¡Tienes razón! Vamos a recapitular todo el proceso desde el principio, enfocándonos en el problema original de ZAP y cómo lo resolvimos.

**Problema Inicial:**

* **ZAP y Reportes HTML:**
    * El contenedor `security` utiliza OWASP ZAP para realizar escaneos de seguridad.
    * ZAP genera reportes en formato HTML, pero no está configurado para servirlos directamente a través de un navegador.
    * Por lo tanto, necesitábamos una forma de acceder a estos reportes HTML desde un navegador.
* **Necesidad de Servir los Reportes:**
    * Decidimos utilizar el contenedor `php` (con Apache) para servir los reportes HTML generados por ZAP.
    * Esto implicaba copiar los reportes desde el contenedor `security` al contenedor `php`.

**Desarrollo y Soluciones:**

1.  **Volumen Compartido:**
    * Para evitar la necesidad de copiar manualmente los archivos, implementamos un volumen compartido (`zap_reports`) entre los contenedores `security` y `php`.
    * Esto permitió que ambos contenedores accedieran al mismo directorio en el sistema de archivos.

2.  **Problemas con la Copia y el Acceso:**
    * Inicialmente, tuvimos dificultades para asegurarnos de que el directorio donde se guardaban los reportes existiera en el contenedor `php` y que Apache pudiera acceder a los archivos.
    * También enfrentamos problemas con los permisos de los archivos y la configuración de Apache.

3.  **Solución Final:**
    * **Configuración del Alias de Apache:**
        * Corregimos la configuración del alias en `zap_reports.conf` para que apuntara a la ruta correcta dentro del volumen compartido: `Alias /zap_reports /zap/reports`.
        * Esto soluciono el problema de que el navegador no encontraba el archivo.
    * **Enlace Simbólico:**
        * Se configuro un enlace simbolico para que la carpeta donde apache sirve los archivos apuntara a la carpeta del volumen compartido.
    * **Automatización:**
        * Se automatizo la creación del directorio y los permisos dentro del contenedor php.
    * **Contenedor auxiliar:**
        * Se creo un contenedor auxiliar para poder ejecutar los comandos de docker dentro del contenedor security, ya que este no tiene instalado docker.

**Pruebas para Verificar la Solución:**

1.  **Verificar la Existencia del Archivo:**
    * Ejecuta `docker exec -it php sh` y luego `ls /zap/reports` para confirmar que el archivo `zap_report.html` está presente.
2.  **Verificar el Enlace Simbólico:**
    * Ejecuta `docker exec -it php sh` y luego `ls -l /var/www/html/zap_reports` para verificar que el enlace simbólico apunta correctamente.
3.  **Verificar la Configuración de Apache:**
    * Ejecuta `docker logs php` para asegurarte de que no haya errores.
4.  **Acceder al Reporte desde el Navegador:**
    * Abre tu navegador y accede a `http://localhost:8080/zap_reports/zap_report.html`.
5.  **Verificar los permisos del archivo:**
    * Ejecuta `docker exec -it php sh` y verifica los permisos del archivo con `ls -l /var/www/html/zap_reports/zap_report.html`. Si los permisos son incorrectos, cambia los permisos con `chmod 644 /var/www/html/zap_reports/zap_report.html`.
6.  **Crear un archivo HTML de prueba:**
    * `docker exec -it php sh`, `echo "<h1>Test</h1>" > /var/www/html/zap_reports/test.html`, y acceder a `http://localhost:8080/zap_reports/test.html`.
